# Big Data Analytics Platform using Apache Spark (PySpark)
### Case Study Solution — Retail Analytics & Recommendation System

**Steps covered:** Spark Init → RDD → Key-Value + Persistence → DataFrames → EDA + Spark SQL → ETL → Machine Learning

Run cells top to bottom. Cell 1 creates the dataset automatically, so nothing extra is needed.

## STEP 0: Generate Synthetic Dataset (customer transactions)

In [ ]:
# Creates synthetic_transactions.csv (5000 records) using plain Python
import csv, random
from datetime import datetime, timedelta

products = {
    "Laptop": ("Electronics", 800), "Mobile": ("Electronics", 500), "Headphones": ("Electronics", 60),
    "TV": ("Electronics", 700), "Shirt": ("Clothing", 25), "Jeans": ("Clothing", 40),
    "Shoes": ("Clothing", 55), "Rice": ("Grocery", 10), "Oil": ("Grocery", 8),
    "Book": ("Stationery", 12), "Pen": ("Stationery", 2), "Sofa": ("Furniture", 400)
}

random.seed(42)
start = datetime(2024, 1, 1)

with open("synthetic_transactions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["transaction_id", "customer_id", "product", "category", "quantity", "price", "purchase_date"])
    for i in range(1, 5001):
        p = random.choice(list(products))
        cat, base = products[p]
        qty = random.randint(1, 5)
        price = round(base * random.uniform(0.9, 1.1), 2)
        date = (start + timedelta(days=random.randint(0, 364))).strftime("%Y-%m-%d")
        w.writerow([i, "C" + str(random.randint(1, 200)), p, cat, qty, price, date])

print("Dataset created: synthetic_transactions.csv (5000 rows)")

## Q1. Spark Initialization and Data Loading (2 Marks)
- **SparkSession** = entry point for DataFrames & SQL
- **SparkContext** = entry point for RDDs (available via `spark.sparkContext`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, month

# Initialize Spark Session
spark = SparkSession.builder.appName("RetailAnalytics").getOrCreate()

# Spark Context (for RDD operations)
sc = spark.sparkContext
print("Spark Version:", spark.version)

# Load the customer transaction dataset
df = spark.read.csv("synthetic_transactions.csv", header=True, inferSchema=True)

# Display schema and sample records
df.printSchema()
df.show(5)
print("Total Records:", df.count())

## Q2. RDD Implementation (3 Marks)
- **Transformations** (lazy): `map`, `filter`
- **Actions** (trigger execution): `take`, `count`, `collect`

In [ ]:
# Create RDD from the DataFrame
rdd = df.rdd

# --- Transformation 1: map -> (customer_id, amount spent) ---
customer_amount = rdd.map(lambda x: (x.customer_id, x.quantity * x.price))

# --- Transformation 2: filter -> only Electronics transactions ---
electronics = rdd.filter(lambda x: x.category == "Electronics")

# --- Actions ---
print("Sample (customer, amount):", customer_amount.take(5))
print("Electronics transactions:", electronics.count())
print("First electronics record:", electronics.first())

## Q3. Key-Value Operations and Persistence (2 Marks)
- `reduceByKey` causes a **shuffle** (data moves across partitions)
- `cache()` / `persist()` stores the RDD in memory so it is not recomputed

In [ ]:
# Key-Value operation: total spend per customer (shuffle happens here)
total_spend = customer_amount.reduceByKey(lambda a, b: a + b)

# Persistence: keep the result in memory
total_spend.cache()

# First action computes and caches, second action reuses the cache
print("Top 5 (customer, total spend):", total_spend.take(5))
print("Number of customers:", total_spend.count())

# sortBy -> another shuffle operation
top_customers_rdd = total_spend.sortBy(lambda x: x[1], ascending=False)
print("Highest spender:", top_customers_rdd.first())

## Q4. Spark DataFrame Operations (3 Marks)
Selection, Filtering, Grouping and Aggregation

In [ ]:
# Selection
df.select("customer_id", "product", "price").show(5)

# Filtering
df.filter(col("category") == "Electronics").show(5)
df.filter(col("price") > 500).show(5)

# Grouping + Aggregation
df.groupBy("category").agg(
    sum(col("quantity") * col("price")).alias("Revenue"),
    count("*").alias("Transactions"),
    avg("price").alias("Avg_Price")
).show()

## Q5. EDA and Spark SQL (5 Marks)
Register the DataFrame as a SQL table, then answer all 5 business questions.

In [ ]:
# Register DataFrame as a temporary SQL view
df.createOrReplaceTempView("sales")

# (a) Customer purchasing patterns
print("--- Customer Purchasing Patterns ---")
spark.sql("""
SELECT customer_id,
       COUNT(*) AS num_transactions,
       ROUND(AVG(quantity*price),2) AS avg_spend
FROM sales
GROUP BY customer_id
ORDER BY num_transactions DESC
LIMIT 10
""").show()

In [ ]:
# (b) Top 10 selling products
print("--- Top 10 Selling Products ---")
spark.sql("""
SELECT product, SUM(quantity) AS total_qty
FROM sales
GROUP BY product
ORDER BY total_qty DESC
LIMIT 10
""").show()

In [ ]:
# (c) Category-wise revenue
print("--- Category-wise Revenue ---")
spark.sql("""
SELECT category, ROUND(SUM(quantity*price),2) AS revenue
FROM sales
GROUP BY category
ORDER BY revenue DESC
""").show()

In [ ]:
# (d) Top 5 customers by total purchase amount
print("--- Top 5 Customers ---")
spark.sql("""
SELECT customer_id, ROUND(SUM(quantity*price),2) AS total_purchase
FROM sales
GROUP BY customer_id
ORDER BY total_purchase DESC
LIMIT 5
""").show()

In [ ]:
# (e) Monthly sales trends
print("--- Monthly Sales Trend ---")
spark.sql("""
SELECT month(purchase_date) AS month,
       ROUND(SUM(quantity*price),2) AS sales
FROM sales
GROUP BY month(purchase_date)
ORDER BY month
""").show()

## Q6. ETL Pipeline (3 Marks)
- **Extract** → read raw CSV
- **Transform** → drop nulls, remove invalid rows, add `amount` column
- **Load** → save cleaned data as CSV and Parquet

In [ ]:
# EXTRACT
raw = spark.read.csv("synthetic_transactions.csv", header=True, inferSchema=True)
print("Extracted rows:", raw.count())

# TRANSFORM
clean = raw.dropna()                                   # remove nulls
clean = clean.filter((col("quantity") > 0) & (col("price") > 0))   # remove invalid rows
clean = clean.withColumn("amount", col("quantity") * col("price")) # new column
print("Rows after cleaning:", clean.count())
clean.show(5)

# LOAD
clean.write.mode("overwrite").csv("processed_sales", header=True)
clean.write.mode("overwrite").parquet("processed_sales_parquet")
print("ETL complete: data saved to processed_sales/ and processed_sales_parquet/")

## Q7. Machine Learning (2 Marks)
Logistic Regression to predict a **high-value purchase** (amount > 500), evaluated with Accuracy and AUC.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Label: 1 if amount > 500 else 0
ml_df = clean.withColumn("label", (col("amount") > 500).cast("int"))

# Features
assembler = VectorAssembler(inputCols=["quantity", "price"], outputCol="features")
data = assembler.transform(ml_df).select("features", "label")

# Train / Test split
train, test = data.randomSplit([0.8, 0.2], seed=42)

# Train model
lr = LogisticRegression()
model = lr.fit(train)

# Predict
pred = model.transform(test)
pred.select("features", "label", "prediction").show(10)

# Evaluate
acc = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred)
auc = BinaryClassificationEvaluator().evaluate(pred)
print("Accuracy:", round(acc, 4))
print("AUC:", round(auc, 4))

## Viva / Quick Notes
- **RDD** = low-level distributed collection (map/filter/reduceByKey)
- **DataFrame** = table-like, optimized by the Catalyst optimizer
- **Shuffle** = data movement across partitions (e.g., reduceByKey, groupBy)
- **cache()/persist()** = keep data in memory to avoid recomputation
- **ETL** = Extract → Transform → Load
- **Logistic Regression** predicts binary outcome (high-value purchase or not)

In [ ]:
# Stop Spark when done
spark.stop()
print("Spark session stopped.")